In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping

In [17]:
import pandas as pd

file_path = r"C:\Users\ADMIN\Documents\mini_project_guvi\project_diamond\regression.csv" 
data = pd.read_csv(file_path)
data.head()

,y,carat,volume,clarity,x,color,price_per_carat,price_inr
0,3.98,0.207014,38.202030,2,3.95,6,27.968908,480.566694
1,3.84,0.190620,34.505856,3,3.89,6,30.374301,480.566694
2,4.07,0.207014,38.076885,5,4.05,6,27.983657,480.820129
3,4.23,0.254642,46.724580,4,4.20,2,22.832547,482.572834
4,4.35,0.270027,51.917250,2,4.34,1,21.542691,482.820226


In [27]:
# Define features and target
selected_features = ["y", "carat", "volume", "clarity", "x", "color", "price_per_carat"]
X = data[selected_features]    # input features
y = data["price_inr"]          # target

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
# Build a simple ANN model
ann_model = Sequential()
ann_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')  # output layer for regression
])

# Compile the model
ann_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


# Train the model
history = ann_model.fit(X_train_scaled, y_train, 
                        validation_split=0.2, 
                        epochs=50, 
                        batch_size=32,
                        callbacks=[early_stop],
                        verbose=1)

Epoch 1/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 112743.2188 - mae: 252.2629 - val_loss: 2032.6943 - val_mae: 35.8353
Epoch 2/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 986.6404 - mae: 24.6500 - val_loss: 746.9197 - val_mae: 20.8895
Epoch 3/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 598.9479 - mae: 19.0003 - val_loss: 525.4910 - val_mae: 17.2967
Epoch 4/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 428.9662 - mae: 15.9800 - val_loss: 393.9694 - val_mae: 14.7375
Epoch 5/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 322.8116 - mae: 13.8100 - val_loss: 306.6214 - val_mae: 12.7577
Epoch 6/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 244.1626 - mae: 11.9277 - val_loss: 244.5937 - val_mae: 11.4102
Epoch 7/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 196.3654 - mae: 10.6904 - val_loss: 201.3092 - val_mae: 10.1808
Epoch 8/50
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 166.1867 - mae: 9.8055 - val_loss: 178.9773 - val_m

In [31]:
# Import metrics (must be done first)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict on test set
y_pred = ann_model.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Print metrics
print(f"MAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.5f}")


337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE: 2.321
MSE: 10.079
RMSE: 3.175
R²: 0.99847


MAE = 2.32 (tiny error)
RMSE = 3.17 (small deviation)
R² = 0.9985 (almost perfect)
Insight: Your ANN is performing exceptionally well on unseen data — the low MAE/RMSE + very high R² is impressive.

In [36]:
# Save the model
ann_model.save("diamond_price_model.keras")
# Later, load it
from tensorflow.keras.models import load_model
loaded_model = load_model("diamond_price_model.keras")

C:\Users\ADMIN\anaconda\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
